In [1]:
import warnings
warnings.filterwarnings("ignore") ## Dọn dẹp và làm sạch màn hình
import pandas as pd ## Thư viện đọc file dữ liệu
import numpy as np ## Thư viện tính toán
import seaborn as sns ## Vẽ các đồ thị
sns.set(style='darkgrid') ## Thiết lập kiểu đồ thị
import matplotlib.pyplot as plt ## Vẽ biểu đồ
%matplotlib inline
import missingno as msno ## Trực quan hóa các giá trị bị thiếu
import os
import glob
import sklearn as sk
import time
seconds = time.time()
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score,precision_recall_fscore_support
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
from xgboost import plot_importance

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
## Đọc dữ liệu từ file đã tối ưu
df = pd.read_csv('/content/drive/MyDrive/BTL_TSL_MR/optimal_data.csv')
## Cột cuối cùng là nhãn (Label)
X = df.drop(columns=[df.columns[-1]])
y = df[df.columns[-1]]
print(f"Tổng số lượng dữ liệu hiện có: {df.shape[0]} dòng")

Tổng số lượng dữ liệu hiện có: 2522362 dòng


In [4]:
## Tách ra 2 tập Train và Test
## Chia 80% để Học (Train) và 20% để Thi (Test)
## Tham số stratify=y giúp giữ nguyên tỷ lệ các loại tấn công
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Kích thước tập Train: {X_train.shape[0]} dòng")
print(f"Kích thước tập Test: {X_test.shape[0]} dòng")
print("\nPhân bố nhãn trên tập Train trước khi cân bằng:")
print(y_train.value_counts())

Kích thước tập Train: 2017889 dòng
Kích thước tập Test: 504473 dòng

Phân bố nhãn trên tập Train trước khi cân bằng:
Attack Type
0    1677187
1     340702
Name: count, dtype: int64


In [5]:
## Chạy Pipeline để cân bằng nhãn
## Do dữ liệu quá lớn nên chạy pipeline để ép tấn công sau đó mới nhân bảng lên để bảo vệ RAM
start_time = time.time()
## Hạ gói tin BENIGN xuống 500.000 dòng
under = RandomUnderSampler(sampling_strategy={0: 500000}, random_state=42)
## SMOTE các gói tin Attack lên mốc 500.000 dòng
over = SMOTE(random_state=42)
steps = [('u', under), ('o', over)]
pipeline = Pipeline(steps=steps)
## Tiến hành cân bằng
X_train_balanced, y_train_balanced = pipeline.fit_resample(X_train, y_train)
print(f"\nHoàn thành cân bằng trong: {round(time.time() - start_time, 2)} giây.")
print("Phân bố nhãn sau khi cân bằng:")
print(pd.Series(y_train_balanced).value_counts())


Hoàn thành cân bằng trong: 764.75 giây.
Phân bố nhãn sau khi cân bằng:
Attack Type
0    500000
1    500000
Name: count, dtype: int64


In [6]:
## Gộp X và y lại để lưu thành 2 file riêng biệt
train_smote_data = pd.concat([X_train_balanced, y_train_balanced], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)
## Lưu vào Drive
train_path = '/content/drive/MyDrive/BTL_TSL_MR/train_smote_data.csv'
test_path = '/content/drive/MyDrive/BTL_TSL_MR/test_data.csv'
train_smote_data.to_csv(train_path, index=False)
test_data.to_csv(test_path, index=False)
print("Đã lưu thành công tập Train và tập Test")

Đã lưu thành công tập Train và tập Test
